# TurboMotors · Do sensor à decisão com Big Data

**Trabalho 02 · Internet das Coisas e suas Aplicações**

Por hora você **não precisa programar**. Todo o código já está pronto. Seu papel é:

1. Executar as células na ordem (clique no ▶ à esquerda de cada uma, ou `Shift + Enter`).
2. Ler o resultado e responder às perguntas do roteiro.
3. Na **Etapa 5 (Simulador)**, trocar os limites nos campos ✏️ e executar de novo.

Atalho: menu **Ambiente de execução → Executar tudo** roda o notebook inteiro de uma vez.

| Etapa | O que fazemos | V do Big Data | Camada IoT |
|---|---|---|---|
| 1 | Carregar telemetria e ordens de serviço | **Volume** e **Velocidade** | Percepção → Rede → Plataforma |
| 2 | Juntar dados do sensor com dados do ERP | **Variedade** | Plataforma + Aplicação (negócio) |
| 3 | Encontrar e limpar leituras erradas | **Veracidade** | Processamento |
| 4 | Resumir por motor e por dia, olhar os gráficos | (prepara o Valor) | Processamento / Análise |
| 5 | Simular a regra de alerta e calcular a economia | **Valor** | Aplicação (decisão) |
| 6 | Comparar regras e escolher a melhor | **Valor** | Aplicação (decisão) |

Os números deste notebook são os mesmos da aba **SIMULADOR** da planilha `turbomotors_telemetria.xlsx`.

## Etapa 0 · Preparar o ambiente

Esta célula carrega as bibliotecas e pede os dois arquivos de dados. Clique em **Escolher arquivos**
e selecione `turbomotors_leituras.csv` e `turbomotors_manutencoes.csv` (os dois de uma vez).

In [ ]:
# @title ▶ Carregar bibliotecas e enviar os dados
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from google.colab import files

plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.max_columns", 20)

# Pede os dois CSV até receber ambos (o Colab pode renomear para "arquivo (1).csv")
enviados = {}
while not ("leituras" in enviados and "manutencoes" in enviados):
    faltam = [n for n in ["leituras", "manutencoes"] if n not in enviados]
    print("Envie: " + " e ".join(f"turbomotors_{n}.csv" for n in faltam))
    for nome, conteudo in files.upload().items():
        for chave in ["leituras", "manutencoes"]:
            if chave in nome:
                enviados[chave] = pd.read_csv(io.BytesIO(conteudo))

leituras = enviados["leituras"]
manutencoes = enviados["manutencoes"]
print("Dados carregados com sucesso!")

## Etapa 1 · Volume e Velocidade

Cada linha de `leituras` é **um motor em uma hora**: temperatura, vibração, corrente e rotação.
Esses dados vêm da cadeia IoT: **sensor (percepção) → gateway/MQTT (rede) → plataforma IoT (ThingsBoard)**,
de onde foram exportados.

In [ ]:
# @title ▶ Ver as primeiras leituras e o tamanho da amostra
print(f"Linhas de telemetria: {len(leituras):,}".replace(",", "."))
print(f"Motores: {leituras['motor_id'].nunique()}  |  Linhas de produção: {leituras['linha'].nunique()}")
print(f"Período: {leituras['data'].min()} a {leituras['data'].max()} ({leituras['data'].nunique()} dias)")
leituras.head(10)

A amostra tem 30 motores e **1 leitura por hora**. Na fábrica real são **500 motores, 4 sensores, leitura a cada 5 segundos**.
Veja a diferença de escala:

In [ ]:
# @title ▶ Calcular o volume da operação real
motores, sensores, intervalo_s = 500, 4, 5
leituras_dia = motores * sensores * (24 * 60 * 60 // intervalo_s)
amostra_dia = leituras["motor_id"].nunique() * 4 * 24

print(f"Operação real:  {leituras_dia:>14,} valores por dia".replace(",", "."))
print(f"Nesta amostra:  {amostra_dia:>14,} valores por dia".replace(",", "."))
print(f"A operação real gera {leituras_dia / amostra_dia:,.0f} vezes mais dados.".replace(",", "."))
print(f"Em 1 ano: {leituras_dia * 365 / 1e9:.1f} bilhões de valores. Isso é Volume. Chegar a cada 5 s é Velocidade.".replace(".", ",", 1))

## Etapa 2 · Variedade

O sensor diz **como o motor está**. Só o **ERP** (sistema de manutenção) diz **quando ele quebrou de verdade**.
Juntar as duas fontes é o que permite testar se uma regra de alerta funciona.

In [ ]:
# @title ▶ Ver as ordens de serviço do ERP
print(manutencoes["tipo"].value_counts().to_string(), "\n")
falhas = manutencoes[manutencoes["tipo"] == "Corretiva"].copy()
falhas["data_falha"] = pd.to_datetime(falhas["inicio"]).dt.normalize()
print("Falhas reais (manutenção corretiva) nos 60 dias:")
falhas[["motor_id", "linha", "data_falha", "descricao", "custo_total_rs"]]

## Etapa 3 · Veracidade

Dados de campo têm erros: sensor desconectado, mensagem duplicada, motor desligado.
Antes de decidir qualquer coisa, precisamos saber **quanto dá para confiar** nos dados.

In [ ]:
# @title ▶ Procurar leituras suspeitas
problemas = pd.Series({
    "Temperatura vazia (sensor não enviou)": leituras["temperatura_c"].isna().sum(),
    "Temperatura impossível (-40 °C ou 9999 °C)": (~leituras["temperatura_c"].between(-10, 150) & leituras["temperatura_c"].notna()).sum(),
    "Vibração impossível (60 mm/s, sensor saturado)": (leituras["vibracao_mm_s"] >= 50).sum(),
    "Motor parado (rotação 0)": (leituras["rotacao_rpm"] == 0).sum(),
    "Leitura duplicada (mesmo motor, mesma hora)": leituras.duplicated(["data_hora", "motor_id"]).sum(),
}, name="quantidade")
problemas.to_frame()

In [ ]:
# @title ▶ Gráfico: um erro de sensor distorce tudo
exemplo = leituras[leituras["motor_id"] == "M207"]
plt.plot(pd.to_datetime(exemplo["data_hora"]), exemplo["temperatura_c"], lw=0.8)
plt.title("Temperatura bruta do motor M207: os picos de 9999 °C e -40 °C são erros do sensor, não do motor")
plt.ylabel("°C")
plt.show()

**Limpeza aplicada** (a mesma da coluna "limpa" da planilha):
remove duplicadas, remove horas com motor parado, e apaga só o valor impossível (temperatura fora de -10 a 150 °C, vibração ≥ 50 mm/s), mantendo os outros sensores daquela hora.

In [ ]:
# @title ▶ Limpar os dados
limpas = leituras.drop_duplicates(["data_hora", "motor_id"])
limpas = limpas[limpas["rotacao_rpm"] > 0].copy()
limpas.loc[~limpas["temperatura_c"].between(-10, 150), "temperatura_c"] = np.nan
limpas.loc[limpas["vibracao_mm_s"] >= 50, "vibracao_mm_s"] = np.nan

print(f"Linhas antes: {len(leituras):,}  |  depois: {len(limpas):,}".replace(",", "."))
print(f"Temperatura máxima antes: {leituras['temperatura_c'].max()} °C  |  depois: {limpas['temperatura_c'].max()} °C")
print(f"Vibração máxima antes: {leituras['vibracao_mm_s'].max()} mm/s  |  depois: {limpas['vibracao_mm_s'].max()} mm/s")

## Etapa 4 · Resumo diário e gráficos

Ninguém decide olhando 43 mil linhas. Resumimos para **1 linha por motor por dia** (igual à aba `diario`):
temperatura **máxima** do dia e vibração **média** do dia.

In [ ]:
# @title ▶ Montar o resumo diário (bruto e limpo)
def resumo_diario(df):
    r = (df.groupby(["data", "motor_id", "linha"])
           .agg(temp_max=("temperatura_c", "max"), vib_media=("vibracao_mm_s", "mean"))
           .round(2).reset_index())
    r["data"] = pd.to_datetime(r["data"])
    return r

diario_bruto = resumo_diario(leituras)
diario_limpo = resumo_diario(limpas)

# Janela de aviso: 14 dias antes da falha + o dia da falha
diario_limpo["janela_falha"] = "-"
for _, f in falhas.iterrows():
    dentro = (diario_limpo["motor_id"] == f["motor_id"]) & diario_limpo["data"].between(f["data_falha"] - pd.Timedelta(days=14), f["data_falha"])
    diario_limpo.loc[dentro, "janela_falha"] = f["motor_id"]
diario_bruto = diario_bruto.merge(diario_limpo[["data", "motor_id", "janela_falha"]], on=["data", "motor_id"])

print(f"{len(leituras):,} leituras viraram {len(diario_limpo):,} linhas de resumo.".replace(",", "."))
diario_limpo.head()

In [ ]:
# @title ▶ Gráfico: vibração média diária. Os motores que falharam aparecem antes da falha?
fig, ax = plt.subplots()
motores_falha = falhas["motor_id"].tolist()
for m, g in diario_limpo.groupby("motor_id"):
    if m not in motores_falha:
        ax.plot(g["data"], g["vib_media"], color="lightgray", lw=0.8)
for _, f in falhas.iterrows():
    g = diario_limpo[diario_limpo["motor_id"] == f["motor_id"]]
    linha, = ax.plot(g["data"], g["vib_media"], lw=2, label=f"{f['motor_id']} (falhou em {f['data_falha']:%d/%m})")
    ax.axvline(f["data_falha"], color=linha.get_color(), ls="--", lw=1)
ax.axhline(4.5, color="red", ls=":", lw=1)
ax.text(diario_limpo["data"].min(), 4.6, "4,5 mm/s: início da zona de alerta (referência)", color="red", fontsize=8)
ax.set_title("Vibração média diária (dados limpos). Cinza = motores sem falha")
ax.set_ylabel("mm/s")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
ax.legend(fontsize=8, loc="upper left")
plt.show()

In [ ]:
# @title ▶ Gráfico: temperatura máxima diária dos mesmos motores
fig, ax = plt.subplots()
for m, g in diario_limpo.groupby("motor_id"):
    if m not in motores_falha:
        ax.plot(g["data"], g["temp_max"], color="lightgray", lw=0.8)
for _, f in falhas.iterrows():
    g = diario_limpo[diario_limpo["motor_id"] == f["motor_id"]]
    linha, = ax.plot(g["data"], g["temp_max"], lw=2, label=f"{f['motor_id']} (falhou em {f['data_falha']:%d/%m})")
    ax.axvline(f["data_falha"], color=linha.get_color(), ls="--", lw=1)
ax.set_title("Temperatura máxima diária (dados limpos). Cinza = motores sem falha")
ax.set_ylabel("°C")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
ax.legend(fontsize=8, loc="upper left")
plt.show()

## Etapa 5 · Simulador da regra de alerta (Valor)

A regra é a mesma da planilha: **alerta** quando a **temperatura máxima do dia** passa do limite
**OU** a **vibração média do dia** passa do limite.

- Uma falha conta como **avisada a tempo** se houve alerta naquele motor em algum dia da janela (14 dias antes até o dia da falha).
- Um dia com alerta fora de qualquer janela de falha é **alarme falso**.
- **Economia** = falhas avisadas × R\$ 35.000 (R\$ 40.000 da corretiva − R\$ 5.000 da troca planejada) − R\$ 1.000 por dia de alarme falso.

✏️ **Troque os valores abaixo e execute a célula de novo.** Use ponto como separador decimal (4.5). Valor **0 desliga** aquele critério.

In [ ]:
# @title ✏️ Limites da regra (altere aqui)
LIMITE_TEMPERATURA = 72  # @param {type:"number"}
LIMITE_VIBRACAO = 0  # @param {type:"number"}
USAR_DADOS_LIMPOS = "NÃO"  # @param ["SIM", "NÃO"]

print(f"Regra: temperatura máxima > {LIMITE_TEMPERATURA or 'desligado'} °C  OU  vibração média > {LIMITE_VIBRACAO or 'desligado'} mm/s  |  dados limpos: {USAR_DADOS_LIMPOS}")

In [ ]:
# @title ▶ Rodar o simulador
def simular(lim_temp, lim_vib, limpos, mostrar=True):
    d = (diario_limpo if limpos == "SIM" else diario_bruto).copy()
    alerta = pd.Series(False, index=d.index)
    if lim_temp:
        alerta |= d["temp_max"].fillna(0) > lim_temp
    if lim_vib:
        alerta |= d["vib_media"].fillna(0) > lim_vib
    d["alerta"] = alerta

    tabela = []
    for _, f in falhas.iterrows():
        a = d[(d["motor_id"] == f["motor_id"]) & (d["janela_falha"] == f["motor_id"]) & d["alerta"]]
        primeiro = a["data"].min() if len(a) else None
        tabela.append({
            "Falha registrada no ERP": f["descricao"],
            "Motor": f["motor_id"],
            "Data": f"{f['data_falha']:%d/%m}",
            "Avisou a tempo?": "SIM" if len(a) else "NÃO",
            "Primeiro alerta": f"{primeiro:%d/%m}" if primeiro is not None else "-",
            "Antecedência (dias)": (f["data_falha"] - primeiro).days if primeiro is not None else "-",
        })
    tabela = pd.DataFrame(tabela)

    avisadas = int((tabela["Avisou a tempo?"] == "SIM").sum())
    falsos = d[d["alerta"] & (d["janela_falha"] == "-")]
    economia = avisadas * 35000 - len(falsos) * 1000
    resultado = {
        "Falhas avisadas a tempo (de 5)": avisadas,
        "Falhas não avisadas (falso negativo)": 5 - avisadas,
        "Dias com alerta sem falha à vista (falso positivo)": len(falsos),
        "Motores com pelo menos um alarme falso": falsos["motor_id"].nunique(),
        "Economia estimada nos 60 dias (R$)": economia,
    }
    if mostrar:
        display(tabela)
        display(pd.Series(resultado, name="Valor").to_frame())
        print(f"\n>>> ECONOMIA: R$ {economia:,.0f}".replace(",", "."))
    return d, resultado

simulado, resultado = simular(LIMITE_TEMPERATURA, LIMITE_VIBRACAO, USAR_DADOS_LIMPOS)

In [ ]:
# @title ▶ Gráfico: onde estão os alarmes falsos?
falsos_por_motor = (simulado[simulado["alerta"] & (simulado["janela_falha"] == "-")]
                    .groupby("motor_id").size().reindex(sorted(simulado["motor_id"].unique()), fill_value=0))
ax = falsos_por_motor.plot.bar(color="indianred")
ax.set_title("Dias de alarme falso por motor com a regra atual")
ax.set_ylabel("dias")
ax.set_xlabel("")
plt.show()

## Etapa 6 · Comparar regras e decidir

Em vez de testar uma regra por vez, coloque várias na lista abaixo e compare lado a lado.
Cada linha é: `(limite de temperatura, limite de vibração, "SIM" ou "NÃO" para dados limpos)`. Use 0 para desligar um critério.

✏️ **Edite a lista**, acrescente as suas regras e execute.

In [ ]:
# ✏️ Regras para comparar (edite esta lista)
regras = [
    (72, 0, "NÃO"),
    (0, 4.5, "NÃO"),
    (72, 4.5, "SIM"),
]

comparacao = []
for t, v, lp in regras:
    _, r = simular(t, v, lp, mostrar=False)
    comparacao.append({"Temp. máx >": t or "-", "Vib. média >": v or "-", "Dados limpos": lp, **r})
comparacao = pd.DataFrame(comparacao).sort_values("Economia estimada nos 60 dias (R$)", ascending=False)
comparacao

### Para fechar (responda no roteiro)

1. Qual regra deu a **maior economia**? Ela usa dados limpos ou brutos? Por quê isso muda o resultado?
2. Alguma falha **nunca** foi avisada, qualquer que fosse o limite? Leia a descrição dela no ERP: um sensor de vibração/temperatura conseguiria prever esse tipo de falha?
3. Quantos dias de **antecedência** a sua melhor regra dá? Isso é suficiente para programar a troca?
4. Se a TurboMotors levar a sua regra para os **500 motores**, em qual camada IoT ela deveria rodar (no gateway, na plataforma ou na aplicação)? Justifique pensando em Volume e Velocidade.